In [1]:
#Install dependencies
from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import requests
from bs4 import BeautifulSoup
import numpy as np

In [2]:
# Set up access to large langauage model as applicable

# Load pre-trained BERT model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)


#Error log
# OSError: [Errno 28] No space left on device.
# -- Action taken: cleared up 5GB on drive C: by moving everything in my downloads folder to drive E:

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [3]:
from transformers import AutoTokenizer, AutoModel

model_name_legal = "nlpaueb/legal-bert-base-uncased"
tokenizer_legal = AutoTokenizer.from_pretrained(model_name_legal)
model_legal = AutoModel.from_pretrained(model_name_legal)


Some weights of the model checkpoint at nlpaueb/legal-bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.bias', 'cls.predictions.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [4]:
#Function to get the text from appropriate source

def get_text_from_policy_doc(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    text = ' '.join([p.get_text() for p in soup.find_all('p')])
    return text 

In [5]:
url = 'https://www.theatlantic.com/privacy-policy/'
url2 = 'https://www.bbc.co.uk/programmes/articles/3VpmkXtXNhkRpQCq0zsWT7T/privacy-notice'
url3 = 'https://www.bbc.co.uk/usingthebbc/privacy-policy/'

In [13]:
# Tokenize, Encode and create chunks

# Define maximum sequence length supported by the model
max_seq_length = 512  # Change according to the model's limitation

#Get text from policy document
policy_document_text = get_text_from_policy_doc(url)

# Tokenize and encode the policy document
encoded_policy = tokenizer_legal.encode_plus(policy_document_text, return_tensors='pt', padding=True, truncation=True)

# Split the encoded policy document into chunks
chunk_size = max_seq_length - 2  # Account for [CLS] and [SEP] tokens
input_ids = encoded_policy['input_ids']
num_chunks = -(-len(input_ids[0]) // chunk_size)  # Ceiling division to determine the number of chunks


In [14]:
policy_document_text

'Date Last Updated: December 18, 2023 The Atlantic Monthly Group LLC and its subsidiaries (referred to throughout this policy as “The Atlantic,” “we,” and “our”) cares about your privacy and the security of your information. This policy details our practices in connection with our print, digital, and live event products and services, including The Atlantic magazine, theatlantic.com, our mobile applications, newsletters, and any other content we provide (collectively, the “Services”) as they relate to your data. It describes the kinds of personal information we may gather about you in connection with the Services, how we use that information, to whom we disclose it, and how you can manage it. By using our Services, you accept the practices described in this Privacy Policy, including our use of cookies and similar online tools\xa0 If you do not agree to the terms of this Privacy Policy, please do not use our Services.  Our services are constantly under development, which may cause our da

In [15]:
# Create Embeddings
# Process each chunk separately

#Empty embeddings list
policy_embeddings = []

for i in range(num_chunks):
    start = i * chunk_size
    end = min((i + 1) * chunk_size, len(input_ids[0]))

    # Get a chunk of input_ids
    input_chunk = input_ids[:, start:end]

    # Get BERT embeddings for the chunk
    with torch.no_grad():
        chunk_outputs = model_legal(input_chunk)
    chunk_embeddings = chunk_outputs.last_hidden_state[:, 0, :]

    policy_embeddings.append(chunk_embeddings)
    
# Concatenate embeddings from all chunks
policy_embeddings = torch.cat(policy_embeddings, dim=0)

In [16]:
# Example conditions obtained from the law (as a dictionary)
law_conditions = {
    "Data Processing Records": "The policy should document how data processing activities are recorded - purposes, data categories, recipients.",
    "Data Subject Rights": "The policy should address and respect data subjects' rights in processing their data.",
    "Consent Mechanisms": "The policy should detail procedures for obtaining, recording, and managing consent for data processing.",
    "Purpose Limitation": "The policy should define lawful purposes for data collection and processing to avoid misuse.",
    "Data Minimisation": "The policy should emphasize collecting only necessary personal data for specified purposes.",
    "Data Security Measures": "The policy should outline security steps -encryption, access controls - for protecting personal data.",
    "Data Breach Response": "The policy should specify procedures for reporting and responding to data breaches.",
    "Data Protection Impact Assessments": "The policy should explain when and how DPIAs are conducted for high-risk activities.",
    "Cross-Border Data Transfers": "The policy should address data protection during international data transfers.",
    "Records of Processing Activities": "The policy should maintain and reference records of processing activities as required.",
    "Data Protection Officer": "The policy should state the appointment of a Data Protection Officer or contact.",
    "Third-Party Data Processors": "The policy should cover use of third-party processors and compliance agreements.",
    "Training and Awareness": "The policy should highlight employee training on data protection importance.",
    "Data Subject Requests": "The policy should outline procedures for handling data subject requests effectively.",
    "Data Retention and Deletion": "The policy should define data retention periods and secure deletion procedures.",
    "Privacy by Design and Default": "The policy should incorporate privacy principles into all data processing.",
    "Updates and Revisions": "The policy should specify how policy updates and revisions are managed and communicated.",
    "Regulatory Reporting": "The policy should address obligations for reporting data processing to authorities.",
    "Incident Documentation": "The policy should mandate documentation of incidents and response actions taken.",
    "Accountability and Governance": "The policy should outline measures demonstrating compliance and governance.",
    "Policy Accessibility": "The policy should ensure easy access to GDPR policy documents for stakeholders.",
    "Legal Basis for Processing": "The policy should provide clear explanations of legal grounds for data processing.",

}



In [10]:
# Calculate similarity scores for each condition

condition_scores = []
results = {}

for criterion, condition in law_conditions.items():  # Iterate over key-value pairs (criterion, condition)
    encoded_condition = tokenizer.encode_plus(condition, return_tensors='pt', padding=True, truncation=True)
    condition_outputs = model(**encoded_condition)
    condition_embeddings = condition_outputs.last_hidden_state[:, 0, :]

    # Calculate cosine similarity between policy document and condition embeddings
    similarity_scores = cosine_similarity(policy_embeddings, condition_embeddings).flatten()

    # Assign scores based on similarity for each chunk
    chunk_scores = [10 if score > 0.8 else 0 for score in similarity_scores]
    condition_score = sum(chunk_scores)
    condition_scores.append(condition_score)
    
    results[criterion] = condition_score
    
    
#ErrorLog
# RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.
# --- Action taken - To resolve this, let's convert the PyTorch tensors 
# --- to NumPy arrays explicitly before passing them into cosine_similarity. 

#     # Convert PyTorch tensors to NumPy arrays
#     policy_numpy = policy_embeddings.detach().cpu().numpy()
#     condition_numpy = condition_embeddings.detach().cpu().numpy()

#     # Calculate cosine similarity between policy document and condition embeddings (using NumPy arrays)
#     similarity_scores = cosine_similarity(policy_numpy, condition_numpy).flatten()


RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.

In [20]:
# Calculate similarity scores for each condition
condition_scores = []
results = {}
similarity_scores= []
chunk_scores = []

for criterion, condition in law_conditions.items():
    encoded_condition = tokenizer_legal.encode_plus(condition, return_tensors='pt', padding=True, truncation=True)
    condition_outputs = model_legal(**encoded_condition)
    condition_embeddings = condition_outputs.last_hidden_state[:, 0, :]

    # Convert PyTorch tensors to NumPy arrays
    policy_numpy = policy_embeddings.detach().cpu().numpy()
    condition_numpy = condition_embeddings.detach().cpu().numpy()

    # Calculate cosine similarity between policy document and condition embeddings (using NumPy arrays)
    similarity_scores = cosine_similarity(policy_numpy, condition_numpy).flatten()

    # Assign scores based on similarity for each chunk
    chunk_scores = [10 if score > 0.49 else 0 for score in similarity_scores]
    condition_score = min(sum(chunk_scores), 10)
    condition_scores.append(condition_score)
    similarity_scores = similarity_scores
    chunk_scores = chunk_scores
    
    results[criterion] = condition_score

In [21]:
#Results for the full 'bbc.co.uk' UK Privacy policy
print(results)

{'Data Processing Records': 10, 'Data Subject Rights': 10, 'Consent Mechanisms': 10, 'Purpose Limitation': 10, 'Data Minimisation': 10, 'Data Security Measures': 10, 'Data Breach Response': 10, 'Data Protection Impact Assessments': 0, 'Cross-Border Data Transfers': 10, 'Records of Processing Activities': 10, 'Data Protection Officer': 10, 'Third-Party Data Processors': 10, 'Training and Awareness': 10, 'Data Subject Requests': 10, 'Data Retention and Deletion': 10, 'Privacy by Design and Default': 10, 'Updates and Revisions': 10, 'Regulatory Reporting': 10, 'Incident Documentation': 10, 'Accountability and Governance': 10, 'Policy Accessibility': 10, 'Legal Basis for Processing': 10}


In [19]:
results

{'Data Processing Records': 10,
 'Data Subject Rights': 10,
 'Consent Mechanisms': 10,
 'Purpose Limitation': 10,
 'Data Minimisation': 10,
 'Data Security Measures': 10,
 'Data Breach Response': 10,
 'Data Protection Impact Assessments': 0,
 'Cross-Border Data Transfers': 10,
 'Records of Processing Activities': 10,
 'Data Protection Officer': 10,
 'Third-Party Data Processors': 10,
 'Training and Awareness': 10,
 'Data Subject Requests': 10,
 'Data Retention and Deletion': 10,
 'Privacy by Design and Default': 10,
 'Updates and Revisions': 10,
 'Regulatory Reporting': 10,
 'Incident Documentation': 10,
 'Accountability and Governance': 10,
 'Policy Accessibility': 10,
 'Legal Basis for Processing': 10}

In [120]:
#Results for 'bbc.co.uk' UK Privacy Notice
print(results)

{'Purpose Limitation': 10, 'Consent Mechanisms': 0, 'Data minimization': 10, 'Right to access and rectification': 10}


In [100]:
#Results for 'TheAtlantic.Com' USA Privacy policy
print(results)

{'Purpose Limitation': 0, 'Consent Mechanisms': 10, 'Data minimization': 10, 'Right to access and rectification': 0}


In [95]:
print(similarity_scores)

[-0.05040023  0.4442248 ]


In [98]:
print(chunk_scores)

[0, 0]


In [110]:
# Calculate the final assessment score by summing up the condition scores
final_score = sum(condition_scores)
print(f"The final assessment score for compliance is: {final_score}")

# Print the scores for each condition with their labels
for criterion, score in results.items():  # Iterate over condition labels from the dictionary
    print(f"{criterion} score: {score}")

The final assessment score for compliance is: 30
Purpose Limitation score: 10
Consent Mechanisms score: 0
Data minimization score: 10
Right to access and rectification score: 10
